# Lab 05 - Linear Systems II

Solutions for all tasks from `L5_NC.pdf`, implemented in Python.

This notebook contains:
1. Jacobi and Gauss-Seidel iterative methods
2. The required large structured-system application
3. Wilson's example for conditioning and perturbation sensitivity
4. The optional SOR application with an `omega` choice guided by the lecture

In [1]:
import numpy as np

np.set_printoptions(precision=6, suppress=True)


def validate_square_matrix(A: np.ndarray) -> np.ndarray:
    A = np.asarray(A, dtype=float)
    if A.ndim != 2 or A.shape[0] != A.shape[1]:
        raise ValueError("A must be a square matrix")
    return A.copy()


def validate_linear_system(A: np.ndarray, b: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    A = validate_square_matrix(A)
    b = np.asarray(b, dtype=float).reshape(-1)
    if b.shape[0] != A.shape[0]:
        raise ValueError("b must have the same length as the dimension of A")
    return A, b.copy()


def residual_norm(A: np.ndarray, x: np.ndarray, b: np.ndarray) -> float:
    A, b = validate_linear_system(A, b)
    x = np.asarray(x, dtype=float).reshape(-1)
    return float(np.linalg.norm(A @ x - b, ord=np.inf))


def relative_vector_error(reference: np.ndarray, perturbed: np.ndarray) -> float:
    reference = np.asarray(reference, dtype=float).reshape(-1)
    perturbed = np.asarray(perturbed, dtype=float).reshape(-1)
    return float(np.linalg.norm(perturbed - reference, ord=np.inf) / np.linalg.norm(reference, ord=np.inf))


def relative_matrix_error(reference: np.ndarray, perturbed: np.ndarray) -> float:
    reference = validate_square_matrix(reference)
    perturbed = validate_square_matrix(perturbed)
    return float(np.linalg.norm(perturbed - reference, ord=np.inf) / np.linalg.norm(reference, ord=np.inf))


def preview_vector(x: np.ndarray, items: int = 4) -> str:
    x = np.asarray(x, dtype=float).reshape(-1)
    if x.size <= 2 * items:
        return np.array2string(x, precision=6, suppress_small=True)
    left = np.array2string(x[:items], precision=6, suppress_small=True)
    right = np.array2string(x[-items:], precision=6, suppress_small=True)
    return f"{left} ... {right}"


def build_tridiagonal_system(n: int) -> tuple[np.ndarray, np.ndarray]:
    if n < 3:
        raise ValueError("n must be at least 3")

    A = 5.0 * np.eye(n)
    idx = np.arange(n - 1)
    A[idx, idx + 1] = -1.0
    A[idx + 1, idx] = -1.0

    b = np.full(n, 3.0)
    b[0] = 4.0
    b[-1] = 4.0
    return A, b


def build_banded_system(n: int) -> tuple[np.ndarray, np.ndarray]:
    if n < 7:
        raise ValueError("n must be at least 7")

    A = 5.0 * np.eye(n)
    for offset in (1, 3):
        idx = np.arange(n - offset)
        A[idx, idx + offset] = -1.0
        A[idx + offset, idx] = -1.0

    b = np.ones(n)
    b[0] = 3.0
    b[1] = 2.0
    b[2] = 2.0
    b[-3] = 2.0
    b[-2] = 2.0
    b[-1] = 3.0
    return A, b


def jacobi(
    A: np.ndarray,
    b: np.ndarray,
    x0: np.ndarray | None = None,
    err: float = 1e-5,
    maxnit: int = 10000,
) -> tuple[np.ndarray, int]:
    A, b = validate_linear_system(A, b)
    n = A.shape[0]
    x = np.zeros(n, dtype=float) if x0 is None else np.asarray(x0, dtype=float).reshape(-1).copy()

    diagonal = np.diag(A)
    if np.any(np.isclose(diagonal, 0.0)):
        raise np.linalg.LinAlgError("Jacobi requires non-zero diagonal entries")

    rest = A - np.diagflat(diagonal)

    for nit in range(1, maxnit + 1):
        x_new = (b - rest @ x) / diagonal
        if np.linalg.norm(x_new - x, ord=np.inf) <= err:
            return x_new, nit
        x = x_new

    raise RuntimeError("Jacobi did not converge within maxnit")


def gauss_seidel(
    A: np.ndarray,
    b: np.ndarray,
    x0: np.ndarray | None = None,
    err: float = 1e-5,
    maxnit: int = 10000,
) -> tuple[np.ndarray, int]:
    A, b = validate_linear_system(A, b)
    n = A.shape[0]
    x = np.zeros(n, dtype=float) if x0 is None else np.asarray(x0, dtype=float).reshape(-1).copy()

    for nit in range(1, maxnit + 1):
        x_old = x.copy()
        for i in range(n):
            left = A[i, :i] @ x[:i]
            right = A[i, i + 1:] @ x_old[i + 1:]
            x[i] = (b[i] - left - right) / A[i, i]

        if np.linalg.norm(x - x_old, ord=np.inf) <= err:
            return x, nit

    raise RuntimeError("Gauss-Seidel did not converge within maxnit")


def sor(
    A: np.ndarray,
    b: np.ndarray,
    omega: float,
    x0: np.ndarray | None = None,
    err: float = 1e-5,
    maxnit: int = 10000,
) -> tuple[np.ndarray, int]:
    if not (0.0 < omega < 2.0):
        raise ValueError("SOR requires 0 < omega < 2")

    A, b = validate_linear_system(A, b)
    n = A.shape[0]
    x = np.zeros(n, dtype=float) if x0 is None else np.asarray(x0, dtype=float).reshape(-1).copy()

    for nit in range(1, maxnit + 1):
        x_old = x.copy()
        for i in range(n):
            left = A[i, :i] @ x[:i]
            right = A[i, i + 1:] @ x_old[i + 1:]
            gs_value = (b[i] - left - right) / A[i, i]
            x[i] = (1.0 - omega) * x_old[i] + omega * gs_value

        if np.linalg.norm(x - x_old, ord=np.inf) <= err:
            return x, nit

    raise RuntimeError("SOR did not converge within maxnit")


def jacobi_iteration_matrix(A: np.ndarray) -> np.ndarray:
    A = validate_square_matrix(A)
    diagonal = np.diag(np.diag(A))
    return np.linalg.solve(diagonal, diagonal - A)


def spectral_radius_power(T: np.ndarray, maxnit: int = 500, tol: float = 1e-12) -> float:
    T = validate_square_matrix(T)
    x = np.ones(T.shape[0], dtype=float)
    x /= np.linalg.norm(x)
    rho_old = 0.0

    for _ in range(maxnit):
        y = T @ x
        rho = np.linalg.norm(y)
        if rho == 0.0:
            return 0.0
        x = y / rho
        if abs(rho - rho_old) <= tol * max(1.0, abs(rho)):
            break
        rho_old = rho

    return float(rho)


def optimal_sor_parameter(A: np.ndarray) -> tuple[float, float]:
    rho_jacobi = spectral_radius_power(jacobi_iteration_matrix(A))
    omega = 2.0 / (1.0 + np.sqrt(1.0 - rho_jacobi**2))
    return float(omega), float(rho_jacobi)

## Exercise 1

For large systems, the lecture notes that direct methods may become too expensive in time or memory, so iterative methods are preferred. Jacobi uses simultaneous replacements, while Gauss-Seidel uses the newest available components and therefore usually converges faster.

For the tridiagonal system from Lab 4,

$$
5x_1 - x_2 = 4, \qquad -x_{j-1} + 5x_j - x_{j+1} = 3 \; (j = 2, \ldots, n-1), \qquad -x_{n-1} + 5x_n = 4,
$$

we have a strictly diagonally dominant matrix, so the lecture convergence criterion applies. We stop when

$$
\|x^{(k)} - x^{(k-1)}\|_\infty \le 10^{-5}.
$$

In [9]:
n = 10
A1, b1 = build_tridiagonal_system(n)
x_expected = np.ones(n)

x_jacobi, nit_jacobi = jacobi(A1, b1, err=0.01, maxnit=1000)
x_gs, nit_gs = gauss_seidel(A1, b1, err=0.01, maxnit=1000)

print(f"n = {n}")
print("Expected solution preview:", preview_vector(x_expected))
print()
print("Jacobi:")
print(f"iterations = {nit_jacobi}")
print("solution preview =", preview_vector(x_jacobi))
print(f"max error vs. ones = {np.linalg.norm(x_jacobi - x_expected, ord=np.inf):.3e}")
print(f"||Ax - b||_inf = {residual_norm(A1, x_jacobi, b1):.3e}")
print()
print("Gauss-Seidel:")
print(f"iterations = {nit_gs}")
print("solution preview =", preview_vector(x_gs))
print(f"max error vs. ones = {np.linalg.norm(x_gs - x_expected, ord=np.inf):.3e}")
print(f"||Ax - b||_inf = {residual_norm(A1, x_gs, b1):.3e}")
print()
print(f"Gauss-Seidel uses {nit_jacobi / nit_gs:.2f}x fewer iterations than Jacobi on this example.")

n = 10
Expected solution preview: [1. 1. 1. 1.] ... [1. 1. 1. 1.]

Jacobi:
iterations = 6
solution preview = [0.99872  0.99776  0.9968   0.996416] ... [0.996416 0.9968   0.99776  0.99872 ]
max error vs. ones = 3.904e-03
||Ax - b||_inf = 1.203e-02

Gauss-Seidel:
iterations = 5
solution preview = [0.999266 0.999083 0.999038 0.999027] ... [0.999679 0.999879 0.999964 0.999993]
max error vs. ones = 9.757e-04
||Ax - b||_inf = 3.249e-03

Gauss-Seidel uses 1.20x fewer iterations than Jacobi on this example.


## Exercise 2 - Wilson's Example

The lecture defines the condition number of a matrix by

$$
\operatorname{cond}(A) = \|A\|\,\|A^{-1}\|.
$$

If `cond(A)` is close to `1`, small relative perturbations in the data produce similarly small relative perturbations in the solution. If `cond(A)` is large, small input changes may create large output changes. The next computation shows exactly that effect.

In [3]:
A2 = np.array([
    [10.0, 7.0, 8.0, 7.0],
    [7.0, 5.0, 6.0, 5.0],
    [8.0, 6.0, 10.0, 9.0],
    [7.0, 5.0, 9.0, 10.0],
])
b2 = np.array([32.0, 23.0, 33.0, 31.0])
b2_perturbed = np.array([32.1, 22.9, 33.1, 30.9])
A2_perturbed = np.array([
    [10.0, 7.0, 8.1, 7.2],
    [7.8, 5.04, 6.0, 5.0],
    [8.0, 5.98, 9.89, 9.0],
    [6.99, 4.99, 9.0, 9.98],
])

x2 = np.linalg.solve(A2, b2)
x2_b = np.linalg.solve(A2, b2_perturbed)
x2_A = np.linalg.solve(A2_perturbed, b2)
cond_inf = np.linalg.cond(A2, np.inf)

print("Original system:")
print("x =", x2)
print(f"||Ax - b||_inf = {residual_norm(A2, x2, b2):.3e}")
print(f"cond_inf(A) = {cond_inf:.3f}")
print()
print("Perturbed right-hand side:")
print("x_tilde =", x2_b)
print(f"relative input error in b = {relative_vector_error(b2, b2_perturbed):.6f}")
print(f"relative output error in x = {relative_vector_error(x2, x2_b):.6f}")
print(f"||A x_tilde - b_tilde||_inf = {residual_norm(A2, x2_b, b2_perturbed):.3e}")
print()
print("Perturbed matrix:")
print("x_hat =", x2_A)
print(f"relative input error in A = {relative_matrix_error(A2, A2_perturbed):.6f}")
print(f"relative output error in x = {relative_vector_error(x2, x2_A):.6f}")
print(f"||A_tilde x_hat - b||_inf = {residual_norm(A2_perturbed, x2_A, b2):.3e}")
print()
print(
    "Explanation: the matrix is ill-conditioned, so small relative perturbations in the data are amplified into much larger relative perturbations in the solution."
)

Original system:
x = [1. 1. 1. 1.]
||Ax - b||_inf = 0.000e+00
cond_inf(A) = 4488.000

Perturbed right-hand side:
x_tilde = [  9.2 -12.6   4.5  -1.1]
relative input error in b = 0.003030
relative output error in x = 13.600000
||A x_tilde - b_tilde||_inf = 7.105e-15

Perturbed matrix:
x_hat = [0.057026 2.365235 0.815666 1.148083]
relative input error in A = 0.025455
relative output error in x = 1.365235
||A_tilde x_hat - b||_inf = 3.553e-15

Explanation: the matrix is ill-conditioned, so small relative perturbations in the data are amplified into much larger relative perturbations in the solution.


## Optional Exercise 3 - SOR

The lecture presents SOR as a relaxation version of Gauss-Seidel:
- `omega = 1` gives Gauss-Seidel;
- `0 < omega < 2` is necessary for convergence;
- for positive definite matrices and `0 < omega < 2`, SOR converges.

It also gives the practical formula

$$
\omega^* = \frac{2}{1 + \sqrt{1 - \rho(T_J)^2}},
$$

where `T_J` is the Jacobi iteration matrix. We estimate `rho(T_J)` with a power iteration and then compare iteration counts.

In [4]:
for n in (10, 1000):
    A3, b3 = build_banded_system(n)
    x_expected = np.ones(n)
    omega_star, rho_jacobi = optimal_sor_parameter(A3)

    x_jacobi, nit_jacobi = jacobi(A3, b3, err=1e-5, maxnit=100000)
    x_gs, nit_gs = gauss_seidel(A3, b3, err=1e-5, maxnit=100000)
    x_sor, nit_sor = sor(A3, b3, omega=omega_star, err=1e-5, maxnit=100000)

    print(f"n = {n}")
    print(f"estimated rho(T_J) = {rho_jacobi:.6f}")
    print(f"omega* = {omega_star:.6f}")
    print(f"Jacobi iterations = {nit_jacobi:>3}, max error = {np.linalg.norm(x_jacobi - x_expected, ord=np.inf):.3e}")
    print(f"Gauss-Seidel iterations = {nit_gs:>3}, max error = {np.linalg.norm(x_gs - x_expected, ord=np.inf):.3e}")
    print(f"SOR iterations = {nit_sor:>3}, max error = {np.linalg.norm(x_sor - x_expected, ord=np.inf):.3e}")
    print()

n = 10
estimated rho(T_J) = 0.676139
omega* = 1.151560
Jacobi iterations =  29, max error = 1.439e-05
Gauss-Seidel iterations =  17, max error = 5.201e-06
SOR iterations =  11, max error = 3.085e-06

n = 1000
estimated rho(T_J) = 0.799949
omega* = 1.249947
Jacobi iterations =  46, max error = 3.484e-05
Gauss-Seidel iterations =  27, max error = 1.760e-05
SOR iterations =  17, max error = 7.640e-06

